<a href="https://colab.research.google.com/github/ndwandweziki-web/JSE-quant-dashboard/blob/main/JSE_Regime_Switching_Quant_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import yfinance as yf

# 1. Download historical daily data from the JSE (Last 5 Years)
print("Fetching JSE market data...")
tickers = ["SBK.JO", "FSR.JO", "^J200.JO"]

# Download without forcing the index filter immediately
raw_data = yf.download(tickers, period="5y", interval="1d")

# Extract close prices safely depending on how yfinance structured the multi-index
if 'Adj Close' in raw_data.columns.levels[0]:
    df_close = raw_data['Adj Close'].copy()
else:
    df_close = raw_data['Close'].copy()

# Drop any rows with missing data to ensure absolute mathematical continuity
df = df_close.dropna().copy()

# 2. Calculate daily log returns
# Formula: R_t = ln(P_t / P_{t-1})
df['SBK_Return'] = np.log(df['SBK.JO'] / df['SBK.JO'].shift(1))
df['FSR_Return'] = np.log(df['FSR.JO'] / df['FSR.JO'].shift(1))
df['JSE_Return'] = np.log(df['^J200.JO'] / df['^J200.JO'].shift(1))

# 3. Create a Rolling Volatility feature for the JSE Index (10-day window)
df['JSE_Vol'] = df['JSE_Return'].rolling(window=10).std()

# Drop the first 10 rows because the rolling volatility creates NaN values
df_features = df.dropna().copy()

print(f"\nData Pipeline Complete. Total trading days captured: {df_features.shape[0]}")
print(df_features[['SBK_Return', 'FSR_Return', 'JSE_Return', 'JSE_Vol']].head())

/tmp/ipykernel_1230/2574026300.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers, period="5y", interval="1d")
[                       0%                       ]

Fetching JSE market data...


[*********************100%***********************]  3 of 3 completed


Data Pipeline Complete. Total trading days captured: 1237
Ticker      SBK_Return  FSR_Return  JSE_Return   JSE_Vol
Date                                                    
2021-07-20    0.025129    0.004591    0.014737  0.018143
2021-07-21   -0.000828   -0.001910    0.013883  0.017970
2021-07-22    0.011851    0.013107    0.013246  0.015763
2021-07-23    0.002125    0.014241    0.010124  0.015073
2021-07-26   -0.013728   -0.010285    0.000216  0.014540


In [5]:
# 1. Isolate our features (Today's market conditions)
X_df = df_features[['FSR_Return', 'JSE_Return', 'JSE_Vol']].copy()

# 2. Isolate the target and shift it BACKWARDS by 1 day
# This pulls tomorrow's return into today's row index so X predicts future y
y_df = df_features['SBK_Return'].shift(-1)

# 3. Drop the very last row because tomorrow's return for today doesn't exist yet
X_df = X_df.iloc[:-1]
y_df = y_df.iloc[:-1]

# 4. Convert to pure NumPy arrays for matrix calculations
X = X_df.values
y = y_df.values

# 5. Manually add the intercept column (column of ones) to the front of X
intercept = np.ones((X.shape[0], 1))
X_mat = np.hstack((intercept, X))

# 6. Write out the OLS Normal Equation formula exactly: beta = (X^T * X)^(-1) * X^T * y
Xt_X_inv = np.linalg.inv(X_mat.T @ X_mat)
beta = Xt_X_inv @ X_mat.T @ y

print(f"Matrix Alignment Complete.")
print(f"Feature Matrix X Shape: {X_mat.shape}")
print(f"Target Vector y Shape:  {y.shape}\n")
print("--- Global Baseline Model Coefficients ---")
print(f"Intercept (c): {beta[0]:.6f}")
print(f"FSR_Return Beta (Slope 1): {beta[1]:.6f}")
print(f"JSE_Return Beta (Slope 2): {beta[2]:.6f}")
print(f"JSE_Vol Beta    (Slope 3): {beta[3]:.6f}")

Matrix Alignment Complete.
Feature Matrix X Shape: (1236, 4)
Target Vector y Shape:  (1236,)

--- Global Baseline Model Coefficients ---
Intercept (c): 0.004749
FSR_Return Beta (Slope 1): -0.043414
JSE_Return Beta (Slope 2): -0.180852
JSE_Vol Beta    (Slope 3): -0.356492


In [6]:
import numpy as np

# 1. Prepare our raw clustering features (JSE_Return, JSE_Vol)
# Shape: (1236, 2)
X_k = X[:, [1, 2]]

# 2. Hardcode the number of clusters (Regimes)
K = 3
np.random.seed(42) # For reproducible random initial centroids

# 3. Initialization Step: Randomly pick K unique days to act as starting cluster centers
random_indices = np.random.choice(X_k.shape[0], K, replace=False)
centroids = X_k[random_indices]

print("Initial Centroids chosen randomly from data:\n", centroids)
print("-" * 50)

# 4. The Iteration Loop (Expectation-Maximization)
max_iters = 100
for iteration in range(max_iters):

    # --- EXPECTATION STEP: Assign each day to the nearest centroid ---
    # We calculate the squared Euclidean distance from every day to all 3 centroids
    distances = np.zeros((X_k.shape[0], K))
    for k in range(K):
        # Euclidean distance formula: sum of squared differences
        distances[:, k] = np.sum((X_k - centroids[k])**2, axis=1)

    # Assign cluster label based on the minimum distance index
    labels = np.argmin(distances, axis=1)

    # --- MAXIMIZATION STEP: Compute new centroids ---
    # The new centroid is the mathematical mean of all days assigned to that cluster
    new_centroids = np.zeros((K, X_k.shape[0] if False else 2)) # Shape (K, 2)
    for k in range(K):
        cluster_days = X_k[labels == k]
        if len(cluster_days) > 0:
            new_centroids[k] = np.mean(cluster_days, axis=0)
        else:
            new_centroids[k] = centroids[k] # Safeguard if a cluster becomes empty

    # Check for Convergence: If centroids stop moving, the optimization is complete
    if np.allclose(centroids, new_centroids):
        print(f"Algorithm converged successfully at iteration {iteration}!")
        centroids = new_centroids
        break

    centroids = new_centroids

# 5. Output the mathematical profiles of our hand-crafted regimes
print("\n--- Final Converged Centroids (Cluster Means) ---")
for k in range(K):
    print(f"Regime {k} -> Avg JSE Return: {centroids[k, 0]:.6f}, Avg JSE Volatility: {centroids[k, 1]:.6f}")
    print(f"          Total days assigned to Regime {k}: {np.sum(labels == k)}")

Initial Centroids chosen randomly from data:
 [[-0.00077376  0.01443479]
 [ 0.01518376  0.01220127]
 [ 0.00656175  0.0149947 ]]
--------------------------------------------------
Algorithm converged successfully at iteration 22!

--- Final Converged Centroids (Cluster Means) ---
Regime 0 -> Avg JSE Return: -0.014382, Avg JSE Volatility: 0.012352
          Total days assigned to Regime 0: 283
Regime 1 -> Avg JSE Return: 0.015806, Avg JSE Volatility: 0.012967
          Total days assigned to Regime 1: 249
Regime 2 -> Avg JSE Return: 0.000975, Avg JSE Volatility: 0.009395
          Total days assigned to Regime 2: 704


In [7]:
import numpy as np

# 1. Prepare inputs: Macro features and the regime labels we generated from K-Means
X_lda = X[:, [1, 2]] # Shape: (1236, 2)
y_lda = labels       # Class labels: 0, 1, 2

n_features = X_lda.shape[1]
class_labels = np.unique(y_lda)

# 2. Calculate the global mean vector
mean_overall = np.mean(X_lda, axis=0)

# 3. Initialize Scatter Matrices
S_W = np.zeros((n_features, n_features))
S_B = np.zeros((n_features, n_features))

# 4. Mathematically populate the matrices
for c in class_labels:
    X_c = X_lda[y_lda == c]
    mean_c = np.mean(X_c, axis=0)

    # Within-Class Scatter: Sum up covariance inside each cluster
    # Formula: (X_c - mean_c)^T * (X_c - mean_c)
    S_W += (X_c - mean_c).T @ (X_c - mean_c)

    # Between-Class Scatter: Distance between cluster means and overall market average
    n_c = X_c.shape[0]
    mean_diff = (mean_c - mean_overall).reshape(n_features, 1)
    S_B += n_c * (mean_diff @ mean_diff.T)

# 5. Solve the optimization problem: Matrix inversion and Eigenvalues
# We want eigenvectors of (S_W^-1 * S_B)
S_W_inv = np.linalg.inv(S_W)
A = S_W_inv @ S_B
eigenvalues, eigenvectors = np.linalg.eig(A)

# Sort eigenvectors by descending eigenvalues to find top discriminants
pairs = [(np.abs(eigenvalues[i]), eigenvectors[:, i]) for i in range(len(eigenvalues))]
pairs.sort(key=lambda x: x[0], reverse=True)

# Create the projection matrix W using the top linear discriminants
W = np.hstack((pairs[0][1].reshape(n_features, 1), pairs[1][1].reshape(n_features, 1)))

print("LDA Training Complete.")
print("Within-Class Scatter Matrix (S_W):\n", S_W)
print("\nBetween-Class Scatter Matrix (S_B):\n", S_B)
print("\nProjection Vector Matrix W (Decision Boundaries Shape):", W.shape)

LDA Training Complete.
Within-Class Scatter Matrix (S_W):
 [[ 0.04268311 -0.0005869 ]
 [-0.0005869   0.02169629]]

Between-Class Scatter Matrix (S_B):
 [[0.12116647 0.0012536 ]
 [0.0012536  0.00323963]]

Projection Vector Matrix W (Decision Boundaries Shape): (2, 2)


In [8]:
# We use our original X_mat (which has the intercept column of ones included)
# Shape: (1236, 4) -> [Intercept, FSR_Return, JSE_Return, JSE_Vol]

print("--- REGIME-SPECIFIC OLS ESTIMATES ---")

for k in range(K):
    # 1. Mathematically isolate the days belonging exclusively to Regime k
    X_k_regime = X_mat[labels == k]
    y_k_regime = y[labels == k]

    # 2. Compute the specialized Normal Equation for this specific market state
    # Formula: beta_k = (X_k^T * X_k)^(-1) * X_k^T * y_k
    XtX_k_inv = np.linalg.inv(X_k_regime.T @ X_k_regime)
    beta_k = XtX_k_inv @ X_k_regime.T @ y_k_regime

    print(f"\n[+] Coefficients for Market Regime {k} ({X_k_regime.shape[0]} trading days):")
    print(f"    Intercept (c):               {beta_k[0]:.6f}")
    print(f"    FSR_Return Beta (Slope 1):    {beta_k[1]:.6f}")
    print(f"    JSE_Return Beta (Slope 2):    {beta_k[2]:.6f}")
    print(f"    JSE_Vol Beta    (Slope 3):    {beta_k[3]:.6f}")

--- REGIME-SPECIFIC OLS ESTIMATES ---

[+] Coefficients for Market Regime 0 (283 trading days):
    Intercept (c):               0.000410
    FSR_Return Beta (Slope 1):    0.084349
    JSE_Return Beta (Slope 2):    -0.151710
    JSE_Vol Beta    (Slope 3):    -0.016687

[+] Coefficients for Market Regime 1 (249 trading days):
    Intercept (c):               -0.010265
    FSR_Return Beta (Slope 1):    -0.581994
    JSE_Return Beta (Slope 2):    4.222162
    JSE_Vol Beta    (Slope 3):    -5.291908

[+] Coefficients for Market Regime 2 (704 trading days):
    Intercept (c):               -0.016263
    FSR_Return Beta (Slope 1):    0.012542
    JSE_Return Beta (Slope 2):    0.546094
    JSE_Vol Beta    (Slope 3):    2.450948


In [9]:
# 1. Take our raw macro features (JSE Return and JSE Volatility)
# Shape: (1236, 2)
X_features = X[:, [1, 2]]

# 2. Mathematically project the 2D features into our LDA space using the projection matrix W
# Linear transformation: X_projected = X * W
# Shape transformation: (1236, 2) @ (2, 2) -> (1236, 2)
X_lda_projected = X_features @ W

# 3. Calculate the projected mean center for each regime in the new LDA space
regime_centers = {}
for k in range(K):
    regime_centers[k] = np.mean(X_lda_projected[labels == k], axis=0)

# 4. Define the live mathematical classification routing function
def route_market_state(today_jse_return, today_jse_vol):
    """
    Projects live daily market metrics into LDA space and routes
    the day to the closest regime center using Euclidean distance.
    """
    # Vectorize the incoming live market data
    live_point = np.array([today_jse_return, today_jse_vol])

    # Project the live point into the optimized LDA space
    projected_point = live_point @ W

    # Calculate distance to all 3 regime centers in the projected space
    distances = []
    for k in range(K):
        dist = np.sum((projected_point - regime_centers[k])**2)
        distances.append(dist)

    # The winner is the regime center with the absolute minimum distance
    predicted_regime = np.argmin(distances)
    return predicted_regime

# 5. Let's test the router with a hypothetical live trading day
# Scenario: The JSE drops by 2% and volatility spikes aggressively to 0.025
test_regime = route_market_state(-0.020000, 0.025000)

print("--- Live Algorithmic Router Initialized ---")
print(f"Projected Regime Centers calculated for all {K} states.")
print(f"Test Scenario Input: JSE Return = -2%, Volatility = 0.025")
print(f"Result: Engine dynamically routed this day to -> **Regime {test_regime}**")

--- Live Algorithmic Router Initialized ---
Projected Regime Centers calculated for all 3 states.
Test Scenario Input: JSE Return = -2%, Volatility = 0.025
Result: Engine dynamically routed this day to -> **Regime 0**


In [10]:
# 1. Map our calculated beta arrays to their respective regimes for clean lookups
# We pull these directly from our specialized OLS loop calculations
regime_betas = {}
for k in range(K):
    X_k_regime = X_mat[labels == k]
    y_k_regime = y[labels == k]
    XtX_k_inv = np.linalg.inv(X_k_regime.T @ X_k_regime)
    regime_betas[k] = XtX_k_inv @ X_k_regime.T @ y_k_regime

# 2. Define the Master Execution Engine
def execute_quant_prediction(today_fsr_return, today_jse_return, today_jse_vol):
    """
    1. Routes the day to the correct market regime using the LDA space.
    2. Pulls the specialized beta coefficients for that regime.
    3. Executes the matrix calculation: y_hat = c + B1*FSR + B2*JSE_Ret + B3*JSE_Vol
    """
    # Step 1: Route to the correct environment using our macro features
    current_regime = route_market_state(today_jse_return, today_jse_vol)

    # Step 2: Extract the corresponding beta weights
    beta_k = regime_betas[current_regime]

    # Step 3: Set up the feature vector with the intercept (1) at the front
    # Feature vector layout: [Intercept, FSR_Return, JSE_Return, JSE_Vol]
    feature_vector = np.array([1.0, today_fsr_return, today_jse_return, today_jse_vol])

    # Step 4: Compute dot product to get the predicted return
    predicted_sbk_return = np.dot(feature_vector, beta_k)

    # Step 5: Generate trading directive based on expected direction
    signal = "BUY / LONG" if predicted_sbk_return > 0 else "SELL / SHORT"

    return current_regime, predicted_sbk_return, signal

# 3. Simulate a live trading day right now
# Scenario: Competitor (FirstRand) drops by 1.5%, JSE Index crashes by 2%, Volatility spikes to 0.025
regime_id, expected_return, trade_directive = execute_quant_prediction(-0.015, -0.020, 0.025)

print("=== QUANT ENGINE LIVE EXECUTION PROTOTYPE ===")
print(f"Dynamically Detected State: Regime {regime_id}")
print(f"Predicted SBK Return for Tomorrow: {expected_return:.6f}")
print(f"Trading Directive Generated:       **{trade_directive}**")

=== QUANT ENGINE LIVE EXECUTION PROTOTYPE ===
Dynamically Detected State: Regime 0
Predicted SBK Return for Tomorrow: 0.001762
Trading Directive Generated:       **BUY / LONG**


In [11]:
import numpy as np
import pandas as pd
import yfinance as yf

def run_production_quant_pipeline():
    print("[1/5] Fetching historical JSE market data...")
    tickers = ["SBK.JO", "FSR.JO", "^J200.JO"]
    raw_data = yf.download(tickers, period="5y", interval="1d")

    df_close = raw_data['Adj Close'].copy() if 'Adj Close' in raw_data.columns.levels[0] else raw_data['Close'].copy()
    df = df_close.dropna().copy()

    # Mathematical log returns & features
    df['SBK_Return'] = np.log(df['SBK.JO'] / df['SBK.JO'].shift(1))
    df['FSR_Return'] = np.log(df['FSR.JO'] / df['FSR.JO'].shift(1))
    df['JSE_Return'] = np.log(df['^J200.JO'] / df['^J200.JO'].shift(1))
    df['JSE_Vol'] = df['JSE_Return'].rolling(window=10).std()
    df_features = df.dropna().copy()

    # Leakage Fix (Shift Target Backwards)
    X_df = df_features[['FSR_Return', 'JSE_Return', 'JSE_Vol']].copy()
    y_df = df_features['SBK_Return'].shift(-1)

    X = X_df.iloc[:-1].values
    y = y_df.iloc[:-1].values

    # Append Intercept
    X_mat = np.hstack((np.ones((X.shape[0], 1)), X))

    print("[2/5] Running Unsupervised K-Means Optimization...")
    X_k = X[:, [1, 2]]
    K = 3
    np.random.seed(42)
    centroids = X_k[np.random.choice(X_k.shape[0], K, replace=False)]

    for _ in range(100):
        distances = np.array([np.sum((X_k - c)**2, axis=1) for c in centroids]).T
        labels = np.argmin(distances, axis=1)
        new_centroids = np.array([np.mean(X_k[labels == k], axis=0) if np.sum(labels == k) > 0 else centroids[k] for k in range(K)])
        if np.allclose(centroids, new_centroids): break
        centroids = new_centroids

    print("[3/5] Running Supervised Linear Discriminant Analysis...")
    mean_overall = np.mean(X_k, axis=0)
    S_W, S_B = np.zeros((2, 2)), np.zeros((2, 2))
    for c in range(K):
        X_c = X_k[labels == c]
        S_W += (X_c - np.mean(X_c, axis=0)).T @ (X_c - np.mean(X_c, axis=0))
        mean_diff = (np.mean(X_c, axis=0) - mean_overall).reshape(2, 1)
        S_B += X_c.shape[0] * (mean_diff @ mean_diff.T)

    eigenvalues, eigenvectors = np.linalg.eig(np.linalg.inv(S_W) @ S_B)
    W = eigenvectors[:, np.argsort(np.abs(eigenvalues))[::-1]]

    X_lda_projected = X_k @ W
    regime_centers = {k: np.mean(X_lda_projected[labels == k], axis=0) for k in range(K)}

    print("[4/5] Computing Regime-Specific OLS Matrices...")
    regime_betas = {}
    for k in range(K):
        X_reg, y_reg = X_mat[labels == k], y[labels == k]
        regime_betas[k] = np.linalg.inv(X_reg.T @ X_reg) @ X_reg.T @ y_reg

    print("[5/5] Engine Fully Compiled. Ready for Real-Time Inference.")
    return W, regime_centers, regime_betas

# Compile the engine variables globally
W, regime_centers, regime_betas = run_production_quant_pipeline()

[1/5] Fetching historical JSE market data...


/tmp/ipykernel_1230/2072476515.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers, period="5y", interval="1d")
[*********************100%***********************]  3 of 3 completed

[2/5] Running Unsupervised K-Means Optimization...
[3/5] Running Supervised Linear Discriminant Analysis...
[4/5] Computing Regime-Specific OLS Matrices...
[5/5] Engine Fully Compiled. Ready for Real-Time Inference.
